In [ ]:
# Copyright 2025 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Document Processing and Entity Extraction with Gemini

This notebook demonstrates document classification and structured entity extraction using Gemini models on both standard high-quality financial reports (Alphabet 10-K, 10-Q) and low-quality/degraded document scans.

## 1. Setup & Document Sample URIs
Imports dependencies and defines Cloud Storage URIs for sample financial documents and degraded scans.

In [ ]:
import os
import sys

# Clone repository and navigate to the entity-extraction folder if running in Colab
if "google.colab" in sys.modules:
    !git clone https://github.com/arieljassan/generative-ai.git
    %cd generative-ai/gemini/use-cases/entity-extraction
    !pip install -q -r requirements.txt

In [ ]:
# Set environment variables, including Project ID and location for running Gemini.

%%writefile .env
# Required: Google Cloud Project ID for Vertex AI Gemini API
GEMINI_PROJECT_ID="my-actual-gcp-project-id"
GEMINI_LOCATION="us-central1"

# Application configuration path
CONFIG_PATH="config.json"

# Evaluation dataset paths (used in evaluate.ipynb)
IMAGE_PATHS="gs://github-repo/generative-ai/gemini/use-cases/entity-extraction/images.csv"
IMAGE_PREFIX="gs://github-repo/generative-ai/gemini/use-cases/entity-extraction"
EVAL_DEST="gs://path/to/evaluations"

# Cloud Run deployment settings (Optional)
CLOUD_RUN_PROJECT_ID="my-actual-gcp-project-id"
CLOUD_RUN_REGION="us-central1"
SERVICE_NAME="entity-extraction-service"
PORT=8080


In [ ]:
# Cloud Storage URIs for sample financial documents (Alphabet Form 10-Q & 10-K)
FORM_10_Q_URI = (
    "gs://cloud-samples-data/gen-app-builder/search/"
    "alphabet-investor-pdfs/2021Q1_alphabet_earnings_release.pdf"
)
FORM_10_K_URI = (
    "gs://cloud-samples-data/gen-app-builder/search/"
    "alphabet-investor-pdfs/2021_alphabet_annual_report.pdf"
)

# Local file path for low-quality (degraded scan) document
FORM_10_Q_DIRTY_URI = "samples/dirty_report.pdf"

In [ ]:
from google.colab import auth

# Authenticate your Google Cloud account in Colab
if "google.colab" in sys.modules:
    auth.authenticate_user()

## 2. Standard Document Classification & Extraction
Classifies document types and extracts structured JSON entities from high-quality document inputs.

In [ ]:
import dotenv

import document_processing

dotenv.load_dotenv(override=True)

In [ ]:
# Classify Form 10-K document type
response = document_processing.classify_document(FORM_10_K_URI)
print(json.loads(response))

## 3. Low-Quality Document Extraction Baseline
Attempts direct entity extraction on a degraded scan without preprocessing to benchmark baseline quality.

In [ ]:
# Attempt classification and extraction directly on low-quality scan
response = (
    document_processing.extract_from_document(
        extract_config_id="form_10_q",
        document_uri=FORM_10_Q_DIRTY_URI
    )
)
data = json.loads(response)
formatted_json_string = json.dumps(data, indent=4)
print(formatted_json_string)

## 4. Enhanced Extraction Pipeline with Quality Assessment
Evaluates visual document quality, applies computer vision preprocessing (denoising, adaptive thresholding), and extracts entities using specialized low-quality prompt templates.

In [ ]:
# Evaluate visual quality, apply image enhancement if needed, and extract
# entities using low-quality prompt template
response = (
    document_processing.evaluate_quality_and_extract(
        extract_config_id="form_10_q",
        document_uri=FORM_10_Q_DIRTY_URI
    )
)
data = json.loads(response)
print(json.dumps(data, indent=4))

## 5. Deploying the solution to Cloud Run
To deploy to Cloud Run, follow the steps in the README file, section "Deploying to Cloud Run"